In [ ]:
import os
import numpy as np
import torch
import timm
from safetensors.torch import load_file
from torchvision.datasets import CIFAR100
from torch.utils.data import Subset

def get_cifar100_subset(data_root, percent, transform=None, seed=42):
    """
    Return a stratified subset of the CIFAR-100 training set with a given percentage.

    Args:
        data_root (str): Directory containing the cifar-100-python folder
        percent (int): Percentage, e.g., 1, 10, 20, 50, 100
        transform (callable, optional): Image transformation to apply to the subset
        seed (int): Random seed for reproducibility

    Returns:
        Subset: torch.utils.data.Subset object that can be directly used in DataLoader
    """
    # Load full training set (no download, assuming it already exists)
    full_train = CIFAR100(root=data_root, train=True, transform=transform, download=False)
    targets = np.array(full_train.targets)
    classes = np.unique(targets)

    # Shuffle indices per class
    np.random.seed(seed)
    indices_per_class = {}
    for c in classes:
        cls_idx = np.where(targets == c)[0]
        np.random.shuffle(cls_idx)
        indices_per_class[c] = cls_idx

    # Take the first n samples per class according to the percentage
    p = percent / 100.0
    subset_idx = []
    for c in classes:
        n_total = len(indices_per_class[c])
        n_select = int(np.ceil(n_total * p)) if p < 1.0 else n_total
        subset_idx.extend(indices_per_class[c][:n_select].tolist())

    # Shuffle globally
    np.random.shuffle(subset_idx)
    return Subset(full_train, subset_idx)


def load_model(model_name, device, num_classes=100):
    """
    Load pretrained model (ConvNeXt-Tiny or ViT-Small/16) and replace the classification head with num_classes.
    """
    if model_name == 'convnext':
        model = timm.create_model('convnext_tiny', pretrained=False, num_classes=num_classes)
        ckpt_path = './models/convnext_tiny.safetensors'
    elif model_name == 'vit':
        model = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=num_classes)
        ckpt_path = './models/vit_small.safetensors'
    else:
        raise ValueError(f"Unknown model: {model_name}")

    # Load safetensors weights
    state_dict = load_file(ckpt_path)

    # Remove all keys related to the classification head (to avoid shape mismatch)
    keys_to_remove = [k for k in state_dict.keys() if k.startswith('head.')]
    for k in keys_to_remove:
        state_dict.pop(k)
        print(f"Removed key: {k}")  # Optional, for debugging

    # Load remaining weights (strict=False allows missing classification head)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    return model

In [2]:
import os
import random
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR100
from torch.cuda.amp import GradScaler, autocast
from safetensors.torch import load_file
import timm

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_cifar100_subset(data_root, percent, transform=None, seed=42):
    full_train = CIFAR100(root=data_root, train=True, transform=transform, download=False)
    targets = np.array(full_train.targets)
    classes = np.unique(targets)

    np.random.seed(seed)
    indices_per_class = {}
    for c in classes:
        cls_idx = np.where(targets == c)[0]
        np.random.shuffle(cls_idx)
        indices_per_class[c] = cls_idx

    p = percent / 100.0
    subset_idx = []
    for c in classes:
        n_total = len(indices_per_class[c])
        n_select = int(np.ceil(n_total * p)) if p < 1.0 else n_total
        subset_idx.extend(indices_per_class[c][:n_select].tolist())

    np.random.shuffle(subset_idx)
    return Subset(full_train, subset_idx)

def load_model(model_name, device, num_classes=100):
    if model_name == 'convnext':
        model = timm.create_model('convnext_tiny', pretrained=False, num_classes=num_classes)
        ckpt_path = './models/convnext_tiny.safetensors'
    elif model_name == 'vit':
        model = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=num_classes)
        ckpt_path = './models/vit_small.safetensors'
    else:
        raise ValueError(f"Unknown model: {model_name}")

    state_dict = load_file(ckpt_path)
    keys_to_remove = [k for k in state_dict.keys() if k.startswith('head.')]
    for k in keys_to_remove:
        state_dict.pop(k)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    return model

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        loss = criterion(outputs, targets)
        total_loss += loss.item() * images.size(0)
        _, pred = outputs.max(1)
        correct += pred.eq(targets).sum().item()
        total += targets.size(0)
    return total_loss / total, correct / total

def find_overfit_epoch(val_losses):
    min_idx = np.argmin(val_losses)
    return min_idx

def run_experiment(model, percent, seed,
                   data_root='./data',
                   output_dir='./results',
                   batch_size=128,
                   epochs=100,
                   lr=1e-4,
                   weight_decay=0.05,
                   warmup_epochs=10,
                   num_workers=4,
                   amp=True):
    """
    Run a single experiment with fixed model, data fraction, and random seed.

    Args:
        model (str): 'convnext' or 'vit'
        percent (int): data percentage (1,10,20,50,100)
        seed (int): random seed (e.g., 42,123,2026)
        data_root (str): CIFAR-100 dataset root directory
        output_dir (str): directory to save results
        batch_size (int): batch size
        epochs (int): number of training epochs
        lr (float): learning rate
        weight_decay (float): weight decay
        warmup_epochs (int): number of warmup epochs
        num_workers (int): number of DataLoader workers
        amp (bool): whether to use mixed precision training

    Returns:
        dict: summary metrics of the experiment
    """
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Create output directory
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Basic data augmentation
    train_transform = transforms.Compose([
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # Fixed validation set
    val_dataset = CIFAR100(root=data_root, train=False, transform=val_transform, download=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)

    exp_id = f"{model}_p{percent}_seed{seed}"
    print(f"\n===== Running: {exp_id} =====")
    set_seed(seed)

    # Training subset
    train_subset = get_cifar100_subset(data_root, percent,
                                       transform=train_transform, seed=seed)
    train_loader = DataLoader(train_subset, batch_size=batch_size,
                              shuffle=True, num_workers=num_workers,
                              pin_memory=True)
    print(f"  Training samples: {len(train_subset)}")

    # Model, loss, optimizer
    model_net = load_model(model, device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model_net.parameters(), lr=lr, weight_decay=weight_decay)

    # Learning rate scheduling: cosine annealing with warmup (per iteration)
    total_steps = epochs * len(train_loader)
    warmup_steps = warmup_epochs * len(train_loader)
    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        else:
            progress = (step - warmup_steps) / (total_steps - warmup_steps)
            return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = GradScaler() if amp else None

    # Record keeping
    epoch_records = []
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    global_step = 0

    for epoch in range(1, epochs + 1):
        model_net.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()

            if amp:
                with autocast():
                    outputs = model_net(images)
                    loss = criterion(outputs, targets)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model_net(images)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()

            scheduler.step()
            global_step += 1

            epoch_loss += loss.item() * images.size(0)
            _, pred = outputs.max(1)
            epoch_correct += pred.eq(targets).sum().item()
            epoch_total += targets.size(0)

        train_loss = epoch_loss / epoch_total
        train_acc = epoch_correct / epoch_total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        val_loss, val_acc = validate(model_net, val_loader, criterion, device)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        epoch_records.append({
            'model': model,
            'percent': percent,
            'seed': seed,
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'lr': scheduler.get_last_lr()[0]
        })

        if epoch % 10 == 0:
            print(f"  Epoch {epoch:3d}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                  f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    # Overfitting analysis
    overfit_epoch = find_overfit_epoch(val_losses)
    final_train_acc = train_accs[-1]
    final_val_acc = val_accs[-1]
    acc_gap = final_train_acc - final_val_acc
    best_val_acc = max(val_accs)
    best_val_epoch = val_accs.index(best_val_acc) + 1

    summary = {
        'model': model,
        'percent': percent,
        'seed': seed,
        'final_train_acc': final_train_acc,
        'final_val_acc': final_val_acc,
        'acc_gap': acc_gap,
        'overfit_epoch': overfit_epoch,
        'best_val_acc': best_val_acc,
        'best_val_epoch': best_val_epoch
    }

    # Save epoch-level details for this experiment
    df_epoch = pd.DataFrame(epoch_records)
    df_epoch.to_csv(Path(output_dir) / f"{exp_id}_epochs.csv", index=False)

    # Save summary (append mode)
    summary_path = Path(output_dir) / "summary_per_run.csv"
    df_summary = pd.DataFrame([summary])
    if summary_path.exists():
        existing = pd.read_csv(summary_path)
        df_summary = pd.concat([existing, df_summary], ignore_index=True)
    df_summary.to_csv(summary_path, index=False)

    print(f"\n===== Experiment {exp_id} finished =====")
    print(f"  Best val acc: {best_val_acc:.4f} at epoch {best_val_epoch}")
    print(f"  Overfit epoch (val loss min): {overfit_epoch}")

    return summary

In [3]:
if __name__ == '__main__':
    # Example: run ConvNeXt, 1% data, seed 42
    # Modify the parameters below to run a specific combination
    result = run_experiment(
        model='vit',   # 'convnext' or 'vit'
        percent=100,          # 1,10,20,50,100
        seed=2026,            # 42,123,2026
        data_root='./data',
        output_dir='./results',
        batch_size=128,
        epochs=100,
        lr=1e-4,
        weight_decay=0.05,
        warmup_epochs=10,
        num_workers=4,
        amp=True  
    )
    print("Experiment result:", result)

Using device: cuda

===== Running: vit_p100_seed2026 =====
  Training samples: 50000


/tmp/ipykernel_2297/2715433115.py:171: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if amp else None
/tmp/ipykernel_2297/2715433115.py:190: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Epoch  10: train_loss=0.1291, train_acc=0.9587, val_loss=0.4570, val_acc=0.8821
  Epoch  20: train_loss=0.0548, train_acc=0.9829, val_loss=0.5563, val_acc=0.8754
  Epoch  30: train_loss=0.0280, train_acc=0.9917, val_loss=0.5911, val_acc=0.8786
  Epoch  40: train_loss=0.0182, train_acc=0.9945, val_loss=0.6021, val_acc=0.8816
  Epoch  50: train_loss=0.0100, train_acc=0.9970, val_loss=0.6491, val_acc=0.8724
  Epoch  60: train_loss=0.0063, train_acc=0.9979, val_loss=0.5995, val_acc=0.8842
  Epoch  70: train_loss=0.0013, train_acc=0.9996, val_loss=0.5902, val_acc=0.8951
  Epoch  80: train_loss=0.0008, train_acc=0.9996, val_loss=0.6254, val_acc=0.8913
  Epoch  90: train_loss=0.0004, train_acc=0.9997, val_loss=0.6010, val_acc=0.8969
  Epoch 100: train_loss=0.0003, train_acc=0.9998, val_loss=0.6017, val_acc=0.8982

===== Experiment vit_p100_seed2026 finished =====
  Best val acc: 0.8988 at epoch 95
  Overfit epoch (val loss min): 4
Experiment result: {'model': 'vit', 'percent': 100, 'seed': 